# 05 — Đánh giá uplift trên final test
**Input:** output NB1–NB4 (version cuối). **GPU:** cần. **Output:** `art/eval/report.md`.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
SCENARIO = "missing"   # missing: base chưa có novel class | weak: base có ít ảnh novel
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"   # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
# luôn lấy bản mới nhất của COMMIT từ GitHub (kể cả khi /tmp/repo còn từ lần chạy trước trong session tương tác)
subprocess.run(["git", "-C", R, "fetch", "-q", "origin", COMMIT], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", "-f", "FETCH_HEAD"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
if f"{R}/src" not in sys.path:
    sys.path.insert(0, f"{R}/src")
for m in [m for m in sys.modules if m == "topicx" or m.startswith("topicx.")]:
    del sys.modules[m]   # chạy lại cell thì nạp lại code mới, không dùng module cũ trong kernel
import numpy as np, pandas as pd
from topicx import common as C, pipeline as P
cfg = C.load_config(SMOKE, SCENARIO)
commit = subprocess.run(["git", "-C", R, "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print("code:", commit)
print(f"kịch bản={SCENARIO} | artifact={cfg['art']} | copy từ input: {C.sync_inputs(cfg)} file")

In [ ]:
STAGE = "eval"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE);

## Health check dataset
Vài giây, không đọc cả file nhãn: đường dẫn, số ảnh, định dạng nhãn, ảnh mẫu có vẽ box. Có mục FAIL thì notebook dừng ở đây.

In [ ]:
hc, sample = P.health_check(cfg)
if sample is not None:
    display(sample.resize((640, 360)))   # box đỏ phải ôm đúng vật thể

In [ ]:
B = cfg["branches"] + cfg["analysis_branches"]
res = P.evaluate(cfg, B, cfg["seeds"], cfg["ks"])
print(f"base mAP50-95: {res['base_map']:.4f} | thiếu: {res['missing'] or 'không'}")
display(res["retrieval"]); display(res["uplift"]); display(res["summary"])

## Kết luận (điền tay)
1. Retrieval có thu thêm positive không?
2. Cùng K, AP novel có tăng không, base mAP giữ được không (so với REPLAY_ONLY)?
3. Còn tăng khi supervision tương đương không (RETRIEVAL_MATCHED)?